In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

df = df.fillna(df.mean()) # dropping is complicated here since features are vague, and a lot of features has a lot of nulls (more than 10K !), so fillna might be a good option here
df

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(exclude=["number"]) # get dt = objects (strings)
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # drop target

scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols])
df.head()

In [ ]:
# Task 5: Write your code here:
df["Target"].value_counts() # to see how many samples of each class

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

accuracy_list = []
f1_list = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # i used StratifiedKFold since target is imbalance

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier( verbose=0,n_estimators=320,  max_depth=4 )


    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy_list.append(accuracy_score(y_test, y_pred))
    f1_list.append(f1_score(y_test, y_pred, zero_division=0))

print(f"accracy mean = {np.mean(accuracy_list):,.2f}")
print(f"f1 mean = {np.mean(f1_list):,.2f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt


# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)[0:5] # to get only the top 5

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = 'P_2'

In [ ]:
# Task Bonus: Write your code here:
X_new = df[[golden_feature, 'D_42']].drop('D_42', axis=1) # adding a column D_42 then dropping it to make the shape (20000, 1) not (20000,)
y_new = df['Target']

accuracy_list_new = []
f1_list_new = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # i used StratifiedKFold since target is imbalance

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_new, y_new), start=1):
    # indexing for each fold
    X_train, X_test = X_new.iloc[train_idx], X_new.iloc[test_idx]
    y_train, y_test = y_new.iloc[train_idx], y_new.iloc[test_idx]

    model = CatBoostClassifier( verbose=0,n_estimators=320,  max_depth=4 )


    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy_list_new.append(accuracy_score(y_test, y_pred))
    f1_list_new.append(f1_score(y_test, y_pred, zero_division=0))

print(f"accracy mean = {np.mean(accuracy_list):,.2f}")
print(f"f1 mean = {np.mean(f1_list):,.2f}")